[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/mlflow-certified/notebooks/day-12-pipelines-preprocessing.ipynb#scrollTo=a3b4c5d6)

---
# Day 12 · Building Preprocessing Pipelines with MLflow and sklearn
**certified-journeys / mlflow-certified** · Practice · Pipelines & Evaluation

> **Goal for today:** Build and log an entire sklearn Pipeline (StandardScaler → PCA → RandomForestClassifier) as a single MLflow model, compare pipeline variants in the UI, and use `mlflow.evaluate()` to get automatic metrics on pipeline output.

In [ ]:
%pip install -q mlflow scikit-learn pandas numpy

## Step 1 · Why Log the Entire Pipeline as One MLflow Model?

A common anti-pattern is to save the preprocessor and the model separately:
```
artifacts/
  scaler.pkl   ← easy to forget, easy to mismatch versions
  model.pkl    ← trained on scaled data, broken without the scaler
```

Logging the full **sklearn Pipeline** as one MLflow model eliminates this:

| Approach | Preprocessing at serve time | Version drift risk |
|---|---|---|
| Separate scaler + model | Manual: caller must run scaler first | High |
| Full pipeline logged | Automatic: pipeline.predict() does it all | None |
| MLflow pyfunc wrapper | Automatic + customizable | None |

The MLflow serving layer calls `pipeline.predict(raw_input)` — raw, unscaled data goes in, predictions come out.

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from mlflow.tracking import MlflowClient
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Use breast cancer dataset — more realistic than Iris (30 features, binary classification)
data = load_breast_cancer(as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42, stratify=data.target
)

# SQLite backend for Model Registry support
mlflow.set_tracking_uri("sqlite:///mlflow_pipeline_demo.db")
mlflow.set_experiment("day-12-pipelines")

print(f"Dataset: {data.data.shape[0]} samples, {data.data.shape[1]} features")
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")
print(f"Features (first 5): {list(data.feature_names[:5])}")
print(f"Classes: {list(data.target_names)}")

**What just happened?**
- We're using the **breast cancer dataset** (30 features) instead of Iris — it's a better showcase for preprocessing because the features are on very different scales.
- **`stratify=data.target`** ensures the train/test split maintains the same class ratio — important for imbalanced datasets.
- With 30 raw features, StandardScaler + PCA will visibly change the data distribution before the classifier sees it.

## Step 2 · Build an sklearn Pipeline and Log It as One MLflow Model

An sklearn `Pipeline` chains transformers and a final estimator:

```
raw input (30 features)
  → StandardScaler      (zero mean, unit variance)
  → PCA                 (reduce to n_components)
  → RandomForestClassifier
  → prediction
```

Key Pipeline properties:
- `pipeline.fit(X, y)` runs `fit_transform` on all transformers, then `fit` on the estimator
- `pipeline.predict(X)` runs `transform` on all transformers, then `predict` on the estimator
- Each step is named (used to set params via `set_params(step__param=value)`)

In [ ]:
def build_rf_pipeline(n_components: int = 10, n_estimators: int = 100) -> Pipeline:
    """Build a StandardScaler → PCA → RandomForest pipeline."""
    return Pipeline([
        ("scaler",     StandardScaler()),                                       # step 1: normalize
        ("pca",        PCA(n_components=n_components, random_state=42)),        # step 2: reduce dims
        ("classifier", RandomForestClassifier(n_estimators=n_estimators,
                                               random_state=42, n_jobs=-1)),    # step 3: classify
    ])

def log_pipeline_run(
    pipeline: Pipeline,
    X_train, y_train, X_test, y_test,
    run_name: str,
    extra_params: dict = None,
) -> str:
    """Fit a pipeline, compute metrics, log everything to MLflow, return run_id."""
    pipeline.fit(X_train, y_train)
    y_pred  = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]

    # Collect all pipeline params for logging
    params = {k: str(v) for k, v in pipeline.get_params().items() if "__" not in k or k.count("__") == 1}
    if extra_params:
        params.update(extra_params)

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "f1":       f1_score(y_test, y_pred),
        "roc_auc":  roc_auc_score(y_test, y_proba),
    }

    sig = infer_signature(X_train, y_pred)

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)

        # Log the entire pipeline — scaler + PCA + classifier bundled together
        mlflow.sklearn.log_model(
            sk_model=pipeline,
            artifact_path="pipeline-model",
            signature=sig,
            input_example=X_train.head(2),
        )
        run_id = run.info.run_id

    print(f"  [{run_name}] accuracy={metrics['accuracy']:.4f} "
          f"f1={metrics['f1']:.4f} roc_auc={metrics['roc_auc']:.4f} "
          f"run_id={run_id[:8]}…")
    return run_id


# Log baseline pipeline
print("Logging pipeline runs…")
run_id_v1 = log_pipeline_run(
    build_rf_pipeline(n_components=10, n_estimators=100),
    X_train, y_train, X_test, y_test,
    run_name="rf-pca10-est100",
)

**What just happened?**
- **`mlflow.sklearn.log_model(pipeline, 'pipeline-model')`** serializes the entire Pipeline object — the scaler's learned mean/scale, the PCA's components, and the forest's trees — all in one artifact.
- **`get_params()`** on a Pipeline returns a flat dict of all params for all steps, enabling easy logging.
- **`infer_signature(X_train, y_pred)`** captures that the model expects a 30-column DataFrame and returns integer labels — the same signature the serving endpoint will enforce.

## Step 3 · Compare Multiple Pipeline Variants in MLflow

The power of MLflow comes from **comparing runs** with different preprocessing configurations. We'll log three variants and then query the best one:

| Run | PCA components | Estimators | Classifier |
|---|---|---|---|
| v1 (baseline) | 10 | 100 | RandomForest |
| v2 | 5 | 200 | RandomForest |
| v3 | 15 | 100 | RandomForest |
| v4 | 10 | 100 | LogisticRegression |

In [ ]:
# Log additional pipeline variants for comparison

# Variant 2: fewer PCA components, more trees
run_id_v2 = log_pipeline_run(
    build_rf_pipeline(n_components=5, n_estimators=200),
    X_train, y_train, X_test, y_test,
    run_name="rf-pca5-est200",
)

# Variant 3: more PCA components
run_id_v3 = log_pipeline_run(
    build_rf_pipeline(n_components=15, n_estimators=100),
    X_train, y_train, X_test, y_test,
    run_name="rf-pca15-est100",
)

# Variant 4: swap RandomForest for LogisticRegression (different classifier, same preprocessor)
lr_pipeline = Pipeline([
    ("scaler",     StandardScaler()),
    ("pca",        PCA(n_components=10, random_state=42)),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])
run_id_v4 = log_pipeline_run(
    lr_pipeline,
    X_train, y_train, X_test, y_test,
    run_name="lr-pca10",
    extra_params={"classifier_type": "LogisticRegression"},
)

print("\nAll 4 pipeline variants logged.")

In [ ]:
# Query and rank all runs by ROC AUC — the programmatic equivalent of the MLflow UI Compare tab
client = MlflowClient()
exp    = client.get_experiment_by_name("day-12-pipelines")

runs = client.search_runs(
    experiment_ids=[exp.experiment_id],
    order_by=["metrics.roc_auc DESC"],
    max_results=10,
)

print(f"{'Run name':<25} {'accuracy':>10} {'f1':>8} {'roc_auc':>10}")
print("-" * 58)
for r in runs:
    name    = r.info.run_name or r.info.run_id[:8]
    acc     = r.data.metrics.get("accuracy", 0)
    f1      = r.data.metrics.get("f1", 0)
    roc_auc = r.data.metrics.get("roc_auc", 0)
    marker  = " ← BEST" if r == runs[0] else ""
    print(f"{name:<25} {acc:>10.4f} {f1:>8.4f} {roc_auc:>10.4f}{marker}")

best_run = runs[0]
best_run_id = best_run.info.run_id
print(f"\nBest run: {best_run.info.run_name} (run_id={best_run_id[:8]}…)")

**What just happened?**
- **`search_runs(order_by=['metrics.roc_auc DESC'])`** translates to a SQL `ORDER BY` — fast, indexed, and works across thousands of runs.
- Logging the **classifier type** as a param lets you distinguish `RandomForest` vs `LogisticRegression` runs in the UI filter bar.
- We identify the best run programmatically — the next step loads it by `run_id` for evaluation.

## Step 4 · Load the Pipeline Model and Run End-to-End Inference on Raw Data

The key test: load the saved pipeline and feed it **raw, unscaled** data. The preprocessing steps should run automatically inside `pipeline.predict()`.

If someone accidentally saved only the classifier (without the scaler), this test would fail or produce garbage predictions — demonstrating why logging the full pipeline matters.

In [ ]:
# Load the best pipeline by run_id
model_uri = f"runs:/{best_run_id}/pipeline-model"
loaded_pipeline = mlflow.sklearn.load_model(model_uri)

print("Loaded pipeline steps:")
for step_name, step_obj in loaded_pipeline.steps:
    print(f"  {step_name}: {type(step_obj).__name__}")

# Confirm the scaler has been fitted (has mean_ attribute)
scaler = loaded_pipeline.named_steps["scaler"]
print(f"\nScaler mean_ (first 3 features): {scaler.mean_[:3].round(3)}")
print(f"Scaler scale_ (first 3 features): {scaler.scale_[:3].round(3)}")

# PCA fitted state
pca = loaded_pipeline.named_steps["pca"]
print(f"\nPCA explained variance ratio (first 3 components): {pca.explained_variance_ratio_[:3].round(4)}")
print(f"Total variance explained: {pca.explained_variance_ratio_.sum():.4f}")

In [ ]:
# End-to-end inference: raw (unscaled) data → predictions
# This proves the scaler and PCA are bundled in the loaded model

# Use raw test data — no manual scaling needed
preds       = loaded_pipeline.predict(X_test)          # raw DataFrame in → labels out
preds_proba = loaded_pipeline.predict_proba(X_test)    # raw DataFrame in → probabilities out

acc     = accuracy_score(y_test, preds)
f1      = f1_score(y_test, preds)
roc_auc = roc_auc_score(y_test, preds_proba[:, 1])

print("End-to-end inference on raw (unscaled) test data:")
print(f"  Accuracy:  {acc:.4f}")
print(f"  F1 score:  {f1:.4f}")
print(f"  ROC AUC:   {roc_auc:.4f}")
print()
print("Sample predictions (first 10):")
print(f"  Predicted: {preds[:10].tolist()}")
print(f"  Actual:    {y_test[:10].tolist()}")
print(f"  Class map: 0={data.target_names[0]}, 1={data.target_names[1]}")

# Verify preprocessing happened: raw feature ranges vs after transform
scaler_step = loaded_pipeline.named_steps["scaler"]
X_scaled = scaler_step.transform(X_test.values)
print(f"\nRaw X_test mean (feat 0): {X_test.iloc[:, 0].mean():.4f}")
print(f"After scaling mean (feat 0): {X_scaled[:, 0].mean():.6f} (≈ 0.0)")

**What just happened?**
- **`loaded_pipeline.predict(X_test)`** internally calls `scaler.transform()` → `pca.transform()` → `classifier.predict()` — all three steps are bundled.
- The scaler state (`mean_`, `scale_`) was saved with the pipeline artifact and is fully restored on load — no need to refit.
- Verifying that the scaled mean is ≈ 0.0 confirms the scaler transform is actually running inside the loaded pipeline.

## Step 5 · Use `mlflow.evaluate()` for Automatic Metric Computation

`mlflow.evaluate()` computes a standard set of metrics for a given model and dataset — no manual `accuracy_score()` calls needed. It also logs the metrics back to an MLflow run.

```python
result = mlflow.evaluate(
    model=model_uri,         # MLflow model URI
    data=eval_df,            # DataFrame with features + label column
    targets="label",         # name of the label column
    model_type="classifier", # 'classifier', 'regressor', or 'question-answering'
)
```

For classifiers, `mlflow.evaluate()` automatically computes:

| Metric | What it measures |
|---|---|
| `accuracy_score` | Fraction of correct predictions |
| `f1_score` | Harmonic mean of precision and recall |
| `precision_score` / `recall_score` | Class-specific performance |
| `roc_auc` | Area under the ROC curve |
| Confusion matrix | Artifact logged as a plot |
| ROC curve | Artifact logged as a plot |

In [ ]:
# Build the evaluation DataFrame: features + label column together
eval_df = X_test.copy()
eval_df["label"] = y_test.values  # mlflow.evaluate needs the label in the DataFrame

print(f"Evaluation DataFrame shape: {eval_df.shape}")
print(f"Columns: {list(eval_df.columns[-5:])}… + label")

In [ ]:
# Run mlflow.evaluate() — logs metrics and artifacts into a new child run
with mlflow.start_run(run_name="evaluate-best-pipeline") as eval_run:
    result = mlflow.evaluate(
        model=model_uri,         # URI of the best pipeline model
        data=eval_df,
        targets="label",
        model_type="classifier",
        evaluators="default",    # built-in sklearn evaluator
    )

print("mlflow.evaluate() results:")
for metric_name, metric_value in sorted(result.metrics.items()):
    print(f"  {metric_name:<35} {metric_value:.4f}")

print(f"\nEval run ID: {eval_run.info.run_id[:8]}…")
print(f"Artifacts logged: {list(result.artifacts.keys())}")

**What just happened?**
- **`mlflow.evaluate()`** loaded the model from the URI, ran inference on `eval_df`, and computed the full metric suite automatically — no manual `sklearn.metrics` calls.
- Metrics are **logged to the MLflow run** (`eval_run`) so they appear in the UI alongside the original training run for comparison.
- **`result.artifacts`** contains paths to generated plots (confusion matrix, ROC curve) that are also logged as MLflow artifacts.
- **`model_type='classifier'`** tells MLflow which metric suite to compute; use `'regressor'` for regression models.

## Step 6 · Cross-Validate a Pipeline and Log Fold Metrics

Cross-validation gives a more reliable estimate of pipeline performance than a single train/test split. We'll log the per-fold metrics as MLflow step metrics — the UI can then plot them as a time series.

```python
mlflow.log_metric("cv_accuracy", fold_acc, step=fold_i)
```

`step` creates a time series in the MLflow UI — each fold is one point on the accuracy curve.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone

# Use the full dataset for cross-validation
X_full = data.data
y_full = data.target

cv_pipeline = build_rf_pipeline(n_components=10, n_estimators=100)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

with mlflow.start_run(run_name="rf-cv5-pca10") as cv_run:
    fold_accs = []
    fold_rocs = []

    for fold_i, (train_idx, val_idx) in enumerate(skf.split(X_full, y_full)):
        X_fold_train = X_full.iloc[train_idx]
        X_fold_val   = X_full.iloc[val_idx]
        y_fold_train = y_full.iloc[train_idx]
        y_fold_val   = y_full.iloc[val_idx]

        # Clone resets the pipeline so each fold starts fresh
        fold_pipe = clone(cv_pipeline)
        fold_pipe.fit(X_fold_train, y_fold_train)

        fold_preds = fold_pipe.predict(X_fold_val)
        fold_proba = fold_pipe.predict_proba(X_fold_val)[:, 1]
        fold_acc   = accuracy_score(y_fold_val, fold_preds)
        fold_roc   = roc_auc_score(y_fold_val, fold_proba)

        fold_accs.append(fold_acc)
        fold_rocs.append(fold_roc)

        # Log per-fold metric as a step (creates a time series in the UI)
        mlflow.log_metric("cv_accuracy", fold_acc, step=fold_i)
        mlflow.log_metric("cv_roc_auc",  fold_roc,  step=fold_i)
        print(f"  Fold {fold_i+1}/5: accuracy={fold_acc:.4f}, roc_auc={fold_roc:.4f}")

    # Log aggregate CV stats
    mlflow.log_metrics({
        "cv_mean_accuracy": np.mean(fold_accs),
        "cv_std_accuracy":  np.std(fold_accs),
        "cv_mean_roc_auc":  np.mean(fold_rocs),
        "cv_std_roc_auc":   np.std(fold_rocs),
    })
    mlflow.log_params({"n_components": 10, "n_estimators": 100, "cv_folds": 5})

print(f"\nCV mean accuracy: {np.mean(fold_accs):.4f} ± {np.std(fold_accs):.4f}")
print(f"CV mean ROC AUC:  {np.mean(fold_rocs):.4f} ± {np.std(fold_rocs):.4f}")
print(f"CV run ID: {cv_run.info.run_id[:8]}…")

**What just happened?**
- **`mlflow.log_metric(name, value, step=fold_i)`** creates a time series — in the MLflow UI, the metric chart shows one point per fold, revealing variance across folds.
- **`sklearn.base.clone`** creates a fresh (unfitted) copy of the pipeline for each fold — essential to avoid data leakage between folds.
- **Mean ± std** gives a better estimate of true performance than a single hold-out split — especially important for small datasets.

## Step 7 · Register the Best Pipeline and Inspect the Artifact

After comparing runs, register the winner to make it available for serving.

In [ ]:
import os, yaml

PIPELINE_MODEL_NAME = "cancer-pipeline"

# Register the best run's pipeline to the Model Registry
mv = mlflow.register_model(
    model_uri=f"runs:/{best_run_id}/pipeline-model",
    name=PIPELINE_MODEL_NAME,
)
print(f"Registered '{PIPELINE_MODEL_NAME}' version {mv.version}")
print(f"  Stage: {mv.current_stage}")

# Set alias for serving
client.set_registered_model_alias(PIPELINE_MODEL_NAME, "champion", mv.version)
print(f"  Alias 'champion' → version {mv.version}")

# Download and inspect the MLmodel manifest to confirm all steps are included
local_path = client.download_artifacts(best_run_id, "pipeline-model", dst_path="/tmp/pipeline_demo")
mlmodel_path = os.path.join(local_path, "MLmodel")

with open(mlmodel_path) as f:
    mlmodel = yaml.safe_load(f)

print("\nMLmodel manifest — flavors:")
for flavor_name, flavor_info in mlmodel.get("flavors", {}).items():
    if flavor_name == "sklearn":
        print(f"  sklearn: sklearn_version={flavor_info.get('sklearn_version')}, "
              f"pickled_model={flavor_info.get('pickled_model')}")
    else:
        print(f"  {flavor_name}: {list(flavor_info.keys())}")

print("\nSignature inputs (first 3):")
for inp in mlmodel.get("signature", {}).get("inputs", [])[:3]:
    print(f"  {inp}")
print(f"  ... and {len(mlmodel.get('signature', {}).get('inputs', [])) - 3} more")

print("\nServing this pipeline:")
print(f"  mlflow models serve -m 'models:/{PIPELINE_MODEL_NAME}@champion' -p 5001 --no-conda")

**What just happened?**
- **`mlflow.register_model`** creates a Registry entry that points to the artifact from the best run — no re-logging needed.
- The **MLmodel manifest** confirms `sklearn` flavor is saved: the `pickled_model` path inside the artifact contains the full Pipeline object.
- The **signature** captures all 30 input column names — the serving endpoint will validate every request against these.
- The serve command is now `models:/cancer-pipeline@champion` — a single alias that always points to the best registered model.

In [ ]:
# Challenge: Build a pipeline comparison function
#
# Write a function `find_best_pipeline(experiment_name, metric='roc_auc')` that:
#   1. Searches all runs in the experiment ordered by the given metric (DESC)
#   2. Returns a dict with keys: run_id, run_name, metrics (all logged metrics),
#      params (all logged params), model_uri
#   3. If no runs are found, raises ValueError with a helpful message
#
# Bonus: Also compute and print the improvement over the median run
#
# Hints:
#   - client.search_runs(experiment_ids=[exp.experiment_id], order_by=[f"metrics.{metric} DESC"])
#   - r.data.metrics, r.data.params, r.info.run_id, r.info.run_name
#   - model_uri = f"runs:/{run_id}/pipeline-model"

def find_best_pipeline(experiment_name: str, metric: str = "roc_auc") -> dict:
    # Your solution here
    pass

# Test it:
# best = find_best_pipeline("day-12-pipelines")
# print("Best pipeline run_id:", best["run_id"][:8])
# print("Best pipeline metrics:", best["metrics"])

---
## Day 12 key concepts recap

| Concept | What to remember |
|---|---|
| Log full Pipeline, not just the model | Scaler + PCA + classifier saved together; no version drift; raw input works at serve time |
| `mlflow.sklearn.log_model(pipeline, ...)` | Works identically for Pipelines and standalone models |
| `pipeline.get_params()` | Returns all params for all steps — log the whole dict at once |
| `sklearn.base.clone` | Creates an unfitted copy — required for CV to avoid leakage across folds |
| `mlflow.log_metric(name, val, step=i)` | Creates a per-fold time series visible in the MLflow UI |
| `mlflow.evaluate()` | Auto-computes full metric suite + plots; logs them to the run |
| `result.metrics` | Dict of all computed metrics after `mlflow.evaluate()` |
| `models:/<name>@champion` | Load or serve by alias — the preprocessing is always bundled |

> **Tip:** Logging the entire sklearn Pipeline as one MLflow model means preprocessing is bundled — there's no separate scaler to manage at serving time.

---
## What's next
**Day 13** → MLflow Projects — package your experiment code into a reproducible, parameterized project runnable with a single `mlflow run` command.

Mark Day 12 complete in your [tracker](../index.html).